## Settings

In [45]:
## auto reload modules
%reload_ext autoreload
%autoreload 2

## Dependencies

In [51]:
## libraries
import sys
import pickle
import pandas as pd
from pathlib import Path
from IPython.display import Markdown, display

## path
root = Path.cwd().resolve().parent
sys.path.insert(0, str(root))

## modules
from src.data.builders import load_processed_data
from src.estimators.factories import load_estimators
from src.evaluators.caching import load_notebook_cache
from src.evaluators.resampling import (
    logo_cross_valid,
    kfold_cross_valid
)
from src.evaluators.transfering import (
    compile_domain_transfer,
    compile_transfer_resampling,
    results_domain_transfer,
    spec_transfer_delta,
    stat_transfer_tost
)
from src.evaluators.predicting import (
    compile_prediction_consensus,
    results_prediction_consensus
)
from src.evaluators.tables import main_table
from src.visualizers.visualize import DEFAULT_MODEL_TO_PARADIGM

## constants
from src.evaluators.config import (
    FEAT_X,
    FEAT_Z, 
    TARGET
)


## Initialization

In [47]:
## reproducibility
N_DECIMALS = 2
N_REPEATS = 30
RANDOM_STATE = 42
FORCE_RECOMPUTE = False

## load data and models
data = load_processed_data()
models = load_estimators(random_state = RANDOM_STATE)

## view data shape
n_obs, n_feat = data.shape
print(f"Original Data: {n_feat} features, {n_obs} observations")

## view model surface
n_mods = len(models)
print(f"Learned Models: {n_mods} estimators")

## cache setup
cache_path = root / "notebooks" / "cache" / "transfer_results.pkl"
cache_metadata = {
    "n_obs": len(data),
    "n_repeats": N_REPEATS,
    "random_state": RANDOM_STATE,
    "model_names": sorted(models.keys()),
    "target": TARGET,
    "feat_x": list(FEAT_X),
    "feat_z": list(FEAT_Z),
}
cache_keys = (
    "predicts_dict_domain",
    "predicts_dict_disc",
    "predicts_dict_5fold",
    "predicts_dict_10fold",
    "results_data_domain",
    "results_data_disc",
    "results_data_5fold",
    "results_data_10fold",
    "results_data_consensus",
)
cache_payload = load_notebook_cache(
    cache_path = cache_path,
    metadata = cache_metadata,
    required_keys = cache_keys,
    force_recompute = FORCE_RECOMPUTE,
)
if cache_payload is not None:
    globals().update({key: cache_payload[key] for key in cache_keys})

Original Data: 32 features, 25 observations
Learned Models: 9 estimators
Loaded results from /Users/nickkunz/Documents/Nick’s MacBook Pro/Python/network-capacity/notebooks/cache/transfer_results.pkl


## Training

In [ ]:
## leave-one-group-out cross validation (domain)
if cache_payload is None:
    results_dict_domain = dict()
    predicts_dict_domain = dict()
    for name, model in models.items():
        print(f"Training {name}...")
        frontier, y_pred = logo_cross_valid(
            data = data,
            feat_x = FEAT_X,
            feat_z = FEAT_Z,
            estimator_c = model.estimator_c,
            estimator_r = model.estimator_r,
            target = TARGET,
            group = "domain",  ## fold on domain
            n_repeats = N_REPEATS,
            random_state = RANDOM_STATE
        )
        frontier["model"] = name
        results_dict_domain[name] = frontier
        predicts_dict_domain[name] = y_pred

In [ ]:
## leave-one-group-out cross validation (discipline)
if cache_payload is None:
    results_dict_disc = dict()
    predicts_dict_disc = dict()
    for name, model in models.items():
        print(f"Training {name}...")
        frontier, y_pred = logo_cross_valid(
            data = data,
            feat_x = FEAT_X,
            feat_z = FEAT_Z,
            estimator_c = model.estimator_c,
            estimator_r = model.estimator_r,
            target = TARGET,
            group = "discipline",  ## fold on discipline
            n_repeats = N_REPEATS,
            random_state = RANDOM_STATE
        )
        frontier["model"] = name
        results_dict_disc[name] = frontier
        predicts_dict_disc[name] = y_pred

In [ ]:
## 5-fold cross validation
if cache_payload is None:
    results_dict_5fold = dict()
    predicts_dict_5fold = dict()
    for name, model in models.items():
        print(f"Training {name}...")
        frontier, y_pred = kfold_cross_valid(
            data = data,
            feat_x = FEAT_X,
            feat_z = FEAT_Z,
            estimator_c = model.estimator_c,
            estimator_r = model.estimator_r,
            target = TARGET,
            n_splits = 5,  ## fold on random 80-20 split
            n_repeats = N_REPEATS,
            random_state = RANDOM_STATE
        )
        frontier["model"] = name
        results_dict_5fold[name] = frontier
        predicts_dict_5fold[name] = y_pred

In [ ]:
## 10-fold cross validation
if cache_payload is None:
    results_dict_10fold = dict()
    predicts_dict_10fold = dict()
    for name, model in models.items():
        print(f"Training {name}...")
        frontier, y_pred = kfold_cross_valid(
            data = data,
            feat_x = FEAT_X,
            feat_z = FEAT_Z,
            estimator_c = model.estimator_c,
            estimator_r = model.estimator_r,
            target = TARGET,
            n_splits = 10,  ## fold on random 90-10 split
            n_repeats = N_REPEATS,
            random_state = RANDOM_STATE
        )
        frontier["model"] = name
        results_dict_10fold[name] = frontier
        predicts_dict_10fold[name] = y_pred

## Post-Processing

In [48]:
## cache setup for transfer results
if cache_payload is None:

    ## compile domain-transfer results
    results_data_domain = compile_domain_transfer(results = results_dict_domain)
    results_data_disc = compile_domain_transfer(results = results_dict_disc)
    results_data_5fold = compile_domain_transfer(results = results_dict_5fold)
    results_data_10fold = compile_domain_transfer(results = results_dict_10fold)

    ## compile prediction consensus (domain logo)
    results_data_consensus = compile_prediction_consensus(
        predictions = predicts_dict_domain,
        data = data,
        target = TARGET,
        group = "domain"
    )

    ## persist canonical transfer results for visualization notebooks
    cache_path.parent.mkdir(parents = True, exist_ok = True)
    cache_payload = {
        "metadata": cache_metadata,
        "predicts_dict_domain": predicts_dict_domain,
        "predicts_dict_disc": predicts_dict_disc,
        "predicts_dict_5fold": predicts_dict_5fold,
        "predicts_dict_10fold": predicts_dict_10fold,
        "results_data_domain": results_data_domain,
        "results_data_disc": results_data_disc,
        "results_data_5fold": results_data_5fold,
        "results_data_10fold": results_data_10fold,
        "results_data_consensus": results_data_consensus,
    }
    cache_path.write_bytes(pickle.dumps(cache_payload))
    print(f"Cached transfer results -> {cache_path}")

## compile common domain-scored resampling surface
results_data_resampling = compile_transfer_resampling(
    predictions = {
        "Domain (LOGO)": predicts_dict_domain,
        "Discipline (LOGO)": predicts_dict_disc,
        "Random (5-Fold)": predicts_dict_5fold,
        "Random (10-Fold)": predicts_dict_10fold,
    },
    data = data,
    target = TARGET,
    group = "domain"
)

## Domain Transfer Test
Four resampling regimes evaluate held-out-group transferability against random-split baselines. Leave-one-group-out (LOGO) cross-validation by domain and discipline assesses generalization to unseen groups; random 5-fold and 10-fold cross-validation establish reference baselines.

- **Domain (LOGO)**: Held-out across 5 domains; group splits fixed, learner initializations vary across repeats.
- **Discipline (LOGO)**: Held-out across 25 disciplines; group splits fixed, learner initializations vary across repeats.
- **Random (5-Fold)**: Repeated shuffled 80–20 splits.
- **Random (10-Fold)**: Repeated shuffled 90–10 splits.

### Reporting Convention
Each regime contributes one value per estimator, averaged across groups or folds and repeats. The summary table reports the median across estimators. `EI` is shown as median `[Q1–Q3]`; `VR`, `MV`, and `MS` report medians, with `MV` and `MS` on the log-target scale. Aggregations are equally weighted across groups, folds, repeats, and estimators.

In [ ]:
## compile transfer bounds and baseline benchmarks
results = results_domain_transfer(
    {"Domain (LOGO)": results_data_domain, "Discipline (LOGO)": results_data_disc},
    {"Random (5-Fold)": results_data_5fold, "Random (10-Fold)": results_data_10fold},
    keys = ("Transfer", "Standard"),
    indicies = ("Evaluation", "Method"),
    n_repeats = N_REPEATS,
    random_state = RANDOM_STATE,
    decimals = N_DECIMALS
)

display(results)

Cross-Validation: 9 models, 30 repeats (seeds 42-71)
Across-model aggregation: median of model-level means
Resampling: LOGO splits are fixed across repeats; random k-fold splits are reshuffled across repeats
Weighting: groups, folds, repeats, and models are equally weighted; results are not observation-weighted


_Transfer `EI` is 0.75 (Domain LOGO) and 0.73 (Discipline LOGO), comparable to the random-split baselines (0.77). Violation rates differ across regimes (0.00–0.04) but remain low throughout, suggesting that estimator performance under strict held-out-group evaluation does not depart materially from random-partition baselines._


## Prediction Consensus Test

The same domain-level LOGO predictions evaluated for frontier efficiency above are scored here for prediction consensus: agreement between the predicted frontier values and the observed maxima of the withheld domain. Frontier estimation is scale-arbitrary across fitting methods, so consensus is measured by agreement in ordering and dependence rather than by residual norms.

- **Spearman (`ρ`)**: Global rank agreement between predicted and observed capacities.
- **Rank-Biased Overlap (`RBO`)**: Top-weighted agreement (persistence $p = 0.9$), emphasizing the high-capacity tail.
- **Distance Correlation (`dCor`)**: Nonlinear dependence between predictions and observations.
- **Consensus Index (`CI`)**: Geometric mean of agreement-scaled `ρ`, `RBO`, and `dCor`.

### Reporting Convention
Each held-out domain contributes one value per estimator. The summary table reports held-out domains row-wise. `CI` is shown as median `[Q1–Q3]` across estimators; component columns report medians. Prior to compounding, `ρ` is rescaled from `[-1, 1]` to `[0, 1]` as `(1 + ρ) / 2` so that negative rank disagreement lowers `CI` without collapsing to a single floor. Aggregations are equally weighted across estimators. Higher values indicate stronger consensus.

In [ ]:
## prediction consensus by held-out domain
results_consensus = results_prediction_consensus(
    results = results_data_consensus,
    group_col = "group",
    index_name = "Domain",
    n_repeats = N_REPEATS,
    random_state = RANDOM_STATE,
    decimals = N_DECIMALS
)

display(results_consensus)

Cross-Validation: 9 models, 30 repeats (seeds 42-71)
Across-model aggregation: median across learners within held-out domains
Resampling: LOGO domain splits are fixed across repeats
Weighting: domains and models are equally weighted; results are not observation-weighted


,CI [IQR],ρ,RBO,dCor
Domain,,,,
Earth & Physical Sciences,"0.78 [0.73, 0.84]",0.60,0.85,0.69
Life Sciences & Medicine,"0.77 [0.77, 0.80]",0.60,0.85,0.69
Technology & Information,"0.85 [0.77, 0.93]",0.90,0.90,0.72
Trade & Institutions,"0.90 [0.84, 0.94]",0.80,0.94,0.88
Transportation & Infrastructure,"0.88 [0.84, 0.96]",0.70,0.86,0.96


_Prediction consensus is consistent across held-out domains. Median `CI` ranges from 0.77 to 0.90, with positive rank agreement (median `ρ` between 0.60 and 0.90) and elevated top-end agreement (`RBO` 0.85–0.94) across all domains. Consensus strength varies by domain but remains positive throughout, indicating that fitted frontiers preserve the observed capacity structure on domains withheld from training._


## Held-Out Group Ranges
Ranges across withheld groups for the frontier efficiency and prediction consensus values reported in the manuscript. Domain-level `EI` follows the Fig. 2a annotation convention, scoring the median frontier across the three learning paradigms within each withheld domain. Discipline-level `EI` and both `CI` rows take the median across estimators within each withheld group, with discipline-level LOGO predictions pooled within domains before scoring `CI`. The pooled `CI` aggregates are medians across all 45 estimator-by-domain cells.


In [52]:
## ranges across withheld groups and pooled consensus for the manuscript
frontier_paradigm = (
    pd.DataFrame(data = predicts_dict_domain).T
    .groupby(by = DEFAULT_MODEL_TO_PARADIGM)
    .median()
    .median(axis = 0)
    .to_numpy()
)
results_data_figure = compile_transfer_resampling(
    predictions = {"Domain (LOGO)": {"paradigm median": frontier_paradigm}},
    data = data,
    target = TARGET,
    group = "domain"
)
results_data_disc_domain = results_data_resampling.loc[
    results_data_resampling["regime"] == "Discipline (LOGO)"
]

ranges = {
    "Domain (LOGO) EI, paradigm-median frontier (Fig. 2a)": results_data_figure["ei"],
    "Discipline (LOGO) EI, median across estimators": results_data_disc.groupby(by = "group", observed = True)["ei"].median(),
    "Domain (LOGO) CI, median across estimators": results_data_consensus.groupby(by = "group", observed = True)["ci"].median(),
    "Discipline (LOGO) CI pooled by domain, median across estimators": results_data_disc_domain.groupby(by = "group", observed = True)["ci"].median(),
}
results_ranges = pd.DataFrame(
    data = {label: {"Min": values.min(), "Max": values.max()} for label, values in ranges.items()}
).T.round(N_DECIMALS)

display(results_ranges)
print(f"Domain (LOGO) CI, median across estimator-by-domain cells: {results_data_consensus['ci'].median():.{N_DECIMALS}f}")
print(f"Discipline (LOGO) CI pooled by domain, median across estimator-by-domain cells: {results_data_disc_domain['ci'].median():.{N_DECIMALS}f}")


,Min,Max
"Domain (LOGO) EI, paradigm-median frontier (Fig. 2a)",0.70,0.91
"Discipline (LOGO) EI, median across estimators",0.52,0.90
"Domain (LOGO) CI, median across estimators",0.77,0.90
"Discipline (LOGO) CI pooled by domain, median across estimators",0.77,0.91


Domain (LOGO) CI, median across estimator-by-domain cells: 0.84
Discipline (LOGO) CI pooled by domain, median across estimator-by-domain cells: 0.84


## Domain Transfer Equivalence Test
The domain transfer test asks whether frontier efficiency (`EI`) under discipline-level LOGO and random-split cross-validation is statistically equivalent to domain-level LOGO. Averaged out-of-fold predictions from all four regimes are rescored within the same five domain subsets, holding the evaluation units fixed while the resampling protocol varies.

- **$H_0$**: The absolute difference meets or exceeds the margin ($|\Delta EI| \ge \delta_{EI}$).
- **$H_1$**: The absolute difference falls strictly within the margin ($|\Delta EI| < \delta_{EI}$).

Rejecting $H_0$ establishes equivalent domain-transfer performance within the prespecified margin.

### Common-Domain Protocol
Each regime contributes one `EI` value per model $\times$ domain. Comparison regimes are paired with Domain LOGO on the same 9 models and 5 domains, yielding $n = 45$ paired values per test. Differences are defined as $\Delta EI = EI_{comparison} - EI_{Domain\ LOGO}$.

### Empirical Test Specification
The equivalence margin is the full interquartile range of model-level Domain LOGO means, with each model first averaged equally across the five domains. Flooring this reference-only dispersion to the second decimal gives $\delta_{EI} = 0.04$. TOST uses paired Wilcoxon signed-rank statistics, and Holm correction is applied across the three regime comparisons.


In [53]:
## resampling regimes compared with domain logo
REGIME_COMPARISONS = (
    "Discipline (LOGO)",
    "Random (5-Fold)",
    "Random (10-Fold)",
)

## empirical domain-transfer margin
delta_ei = spec_transfer_delta(
    results = results_data_resampling,
    metric = "ei",
    reference = "Domain (LOGO)",
    label_regime = "regime",
    model_col = "model",
    scale = 1.0,
    decimals = N_DECIMALS
)

## paired tost across resampling regimes
results_transfer_ei = stat_transfer_tost(
    results = results_data_resampling,
    metric = "ei",
    delta = delta_ei,
    reference = "Domain (LOGO)",
    comparisons = REGIME_COMPARISONS,
    feat_pairs = ("model", "group"),
    label_regime = "regime",
    decimals = N_DECIMALS,
    index = True
)

display(results_transfer_ei)

Paired TOST (Wilcoxon Signed-Rank): n = 45, δ = 0.04, metric = EI
H₀: |Δ EI| ≥ δ
H₁: |Δ EI| < δ
Median Δ EI: Median of paired differences (comparison - Domain (LOGO))
Rank-biserial r: Raw paired effect size. Positive values indicate comparison > Domain (LOGO)
TOST p: max(Upper p, Lower p)
Holm-adj. p: Holm-Bonferroni adjusted across regime comparisons
Significance codes reflect Holm-adj. p
*** p < 0.001, ** p < 0.01, * p < 0.05


,Median Δ EI,Rank-biserial r,TOST p,Holm-adj. p,Sig.,Eq.
Regime,,,,,,
Discipline (LOGO),0.00,-0.11,<0.001,<0.001,***,Yes
Random (5-Fold),0.00,0.35,<0.001,<0.001,***,Yes
Random (10-Fold),0.00,0.08,<0.001,<0.001,***,Yes


_All three resampling regimes satisfy equivalence to Domain LOGO for frontier efficiency within $\delta_{EI} = 0.04$. Median shifts round to 0.00, with all Holm-adjusted TOST $p$-values below 0.001, supporting equivalent domain-transfer performance across held-out-group and random-split evaluation._


## Prediction Consensus Equivalence Test
The prediction consensus test asks whether each alternative resampling regime preserves the same domain-level capacity ordering as Domain LOGO. Consensus is measured by the Consensus Index $CI = f(\rho, RBO, dCor)$ after scoring each regime's averaged out-of-fold predictions within the same five domains.

- **$H_0$**: The absolute difference meets or exceeds the margin ($|\Delta CI| \ge \delta_{CI}$).
- **$H_1$**: The absolute difference falls strictly within the margin ($|\Delta CI| < \delta_{CI}$).

Rejecting $H_0$ establishes equivalent prediction consensus within the prespecified margin.

### Common-Domain Protocol
Each regime contributes one `CI` value per model $\times$ domain. Comparison regimes are paired with Domain LOGO on the same 45 evaluation units used by the `EI` test. Differences are defined as $\Delta CI = CI_{comparison} - CI_{Domain\ LOGO}$.

### Empirical Test Specification
The equivalence margin is the full interquartile range of model-level Domain LOGO means, with each model first averaged equally across the five domains. Flooring this reference-only dispersion to the second decimal gives $\delta_{CI} = 0.11$. TOST uses paired Wilcoxon signed-rank statistics, and Holm correction is applied across the three regime comparisons.


In [54]:
## empirical prediction-consensus margin
delta_ci = spec_transfer_delta(
    results = results_data_resampling,
    metric = "ci",
    reference = "Domain (LOGO)",
    label_regime = "regime",
    model_col = "model",
    scale = 1.0,
    decimals = N_DECIMALS
)

## paired tost across resampling regimes
results_transfer_ci = stat_transfer_tost(
    results = results_data_resampling,
    metric = "ci",
    delta = delta_ci,
    reference = "Domain (LOGO)",
    comparisons = REGIME_COMPARISONS,
    feat_pairs = ("model", "group"),
    label_regime = "regime",
    decimals = N_DECIMALS,
    index = True
)

display(results_transfer_ci)

Paired TOST (Wilcoxon Signed-Rank): n = 45, δ = 0.11, metric = CI
H₀: |Δ CI| ≥ δ
H₁: |Δ CI| < δ
Median Δ CI: Median of paired differences (comparison - Domain (LOGO))
Rank-biserial r: Raw paired effect size. Positive values indicate comparison > Domain (LOGO)
TOST p: max(Upper p, Lower p)
Holm-adj. p: Holm-Bonferroni adjusted across regime comparisons
Significance codes reflect Holm-adj. p
*** p < 0.001, ** p < 0.01, * p < 0.05


,Median Δ CI,Rank-biserial r,TOST p,Holm-adj. p,Sig.,Eq.
Regime,,,,,,
Discipline (LOGO),0.00,-0.23,<0.001,<0.001,***,Yes
Random (5-Fold),-0.02,-0.54,<0.001,<0.001,***,Yes
Random (10-Fold),-0.01,-0.55,<0.001,<0.001,***,Yes


_All three resampling regimes satisfy equivalence to Domain LOGO for prediction consensus within $\delta_{CI} = 0.11$. Median `CI` shifts are 0.02 at most in magnitude, and all Holm-adjusted TOST $p$-values are below 0.001. The fitted frontiers therefore preserve equivalent domain-level ordering and dependence across held-out-group and random-split evaluation._


## Summary

In [55]:
## manuscript summary table for resampling equivalence
display_table = main_table(
    results_left = results_transfer_ei.reset_index(),
    results_right = results_transfer_ci.reset_index(),
    index_feat = ["Regime"],
    value_feat_left = ["Median Δ EI", "Rank-biserial r", "Sig.", "Eq."],
    value_feat_right = ["Median Δ CI", "Rank-biserial r", "Sig.", "Eq."],
    header_left = "Domain Transfer",
    header_right = "Prediction Consensus",
    short_feat_left = ["Δ EI", "r", "Sig.", "Eq."],
    short_feat_right = ["Δ CI", "r", "Sig.", "Eq."],
    row_order = [
        ("Discipline (LOGO)",),
        ("Random (5-Fold)",),
        ("Random (10-Fold)",),
    ],
)

display(display_table)
display(
    Markdown(
        "**Δ EI**, **Δ CI**: median paired difference between the listed comparison regime and Domain LOGO. "
        "Positive values indicate higher scores under the comparison regime. "
        "**r**: paired rank-biserial correlation effect size. "
        "**Sig.**: Holm-adjusted TOST *p* below threshold within each outcome "
        "(\*\*\* *p* < 0.001, \*\* *p* < 0.01, \* *p* < 0.05). "
        "Significance supports the equivalence hypothesis H₁: |Δ| < δ. "
        "**Eq.**: equivalence supported when Holm-adjusted TOST *p* < 0.05. "
        f"Margins are full IQRs of model-level Domain LOGO means, with δ_EI = {delta_ei:.{N_DECIMALS}f} and δ_CI = {delta_ci:.{N_DECIMALS}f}. "
        "Each test uses *n* = 45 paired values from 9 models × 5 domains."
    )
)


Domain Transfer                  Prediction Consensus  \
                             Δ EI      r Sig.  Eq.                 Δ CI   
Regime                                                                    
Discipline (LOGO)            0.00  -0.11  ***  Yes                 0.00   
Random (5-Fold)              0.00   0.35  ***  Yes                -0.02   
Random (10-Fold)             0.00   0.08  ***  Yes                -0.01   

                                    
                       r Sig.  Eq.  
Regime                              
Discipline (LOGO)  -0.23  ***  Yes  
Random (5-Fold)    -0.54  ***  Yes  
Random (10-Fold)   -0.55  ***  Yes

**Δ EI**, **Δ CI**: median paired difference between the listed comparison regime and Domain LOGO. Positive values indicate higher scores under the comparison regime. **r**: paired rank-biserial correlation effect size. **Sig.**: Holm-adjusted TOST *p* below threshold within each outcome (\*\*\* *p* < 0.001, \*\* *p* < 0.01, \* *p* < 0.05). Significance supports the equivalence hypothesis H₁: |Δ| < δ. **Eq.**: equivalence supported when Holm-adjusted TOST *p* < 0.05. Margins are full IQRs of model-level Domain LOGO means, with δ_EI = 0.04 and δ_CI = 0.11. Each test uses *n* = 45 paired values from 9 models × 5 domains.